# 01 — Explorar Docling com 1 PDF real

**Objetivo:** Entender a API Docling factualmente. Sem col_map, sem Aho-Corasick, sem pipeline.

**PDF:** AGU — Anexo de Entregas (PTD 2025-2027)

**Ao final sabemos:** quais atributos `doc` tem, como acessar tabelas, como acessar proveniencia.

## 1. Setup minimo

In [ ]:
!pip install -q docling docling-core

## 2. Baixar 1 PDF real

In [ ]:
!wget -q -O agu_entregas.pdf \
  'https://www.gov.br/governodigital/pt-br/estrategias-e-governanca-digital/planos-de-transformacao-digital/ptds-vigentes/agu_ptd_2025_2027_anexo_de_entregas_assinado_assinado_assinado-copia_tarjada.pdf'

import os
size = os.path.getsize('agu_entregas.pdf')
print(f'PDF baixado: {size:,} bytes ({size/1024:.0f} KB)')

## 3. Converter com Docling

In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert('agu_entregas.pdf')
doc = result.document

print(f'Tipo do doc: {type(doc).__name__}')
print(f'Conversao OK')

## 4. Explorar: o que `doc` tem?

In [ ]:
# Atributos publicos (sem _)
attrs = [a for a in dir(doc) if not a.startswith('_')]
print(f'Atributos de doc ({len(attrs)}):')
for a in sorted(attrs):
    val = getattr(doc, a, '???')
    tipo = type(val).__name__
    if callable(val):
        print(f'  {a}() — metodo')
    elif isinstance(val, (list, dict, tuple)):
        print(f'  {a} — {tipo}, len={len(val)}')
    else:
        print(f'  {a} — {tipo}')

## 5. Acessar tabelas

In [ ]:
# Tentar multiplos caminhos conhecidos da API Docling
print('=== Tentando acessar tabelas ===')

# Caminho 1: doc.tables (usado no nosso ptd_extracao.py)
if hasattr(doc, 'tables'):
    tables = doc.tables
    print(f'doc.tables existe! Tipo: {type(tables).__name__}, len: {len(tables) if hasattr(tables, "__len__") else "?"}')
else:
    print('doc.tables NAO existe')

# Caminho 2: iterar main_text procurando TableItem
if hasattr(doc, 'main_text'):
    from collections import Counter
    tipos = Counter(type(item).__name__ for item in doc.main_text)
    print(f'\ndoc.main_text existe! {len(doc.main_text)} itens')
    print(f'Tipos: {dict(tipos)}')
else:
    print('doc.main_text NAO existe')

# Caminho 3: doc.body
if hasattr(doc, 'body'):
    print(f'\ndoc.body existe! Tipo: {type(doc.body).__name__}')
else:
    print('doc.body NAO existe')

# Caminho 4: iterate_items
if hasattr(doc, 'iterate_items'):
    from collections import Counter
    tipos2 = Counter(type(item).__name__ for item, _ in doc.iterate_items())
    print(f'\ndoc.iterate_items() existe!')
    print(f'Tipos: {dict(tipos2)}')
else:
    print('doc.iterate_items NAO existe')

## 6. Explorar 1 tabela em detalhe

In [ ]:
# Pegar a primeira tabela por qualquer caminho que funcionou
table = None

if hasattr(doc, 'tables') and doc.tables:
    if isinstance(doc.tables, dict):
        key = list(doc.tables.keys())[0]
        table = doc.tables[key]
        print(f'Tabela via doc.tables[{key!r}]')
    elif isinstance(doc.tables, list):
        table = doc.tables[0]
        print(f'Tabela via doc.tables[0]')

if table is None and hasattr(doc, 'iterate_items'):
    for item, level in doc.iterate_items():
        if 'Table' in type(item).__name__:
            table = item
            print(f'Tabela via iterate_items(): {type(item).__name__}')
            break

if table is None:
    print('NENHUMA TABELA ENCONTRADA — verificar output da celula 5')
else:
    print(f'\nTipo: {type(table).__name__}')
    print(f'Atributos:')
    for a in sorted(dir(table)):
        if not a.startswith('_'):
            val = getattr(table, a, '???')
            print(f'  {a} — {type(val).__name__}')

## 7. Proveniencia (ProvenanceItem)

In [ ]:
if table is None:
    print('Sem tabela — pular')
elif hasattr(table, 'prov') and table.prov:
    prov = table.prov[0]
    print(f'ProvenanceItem encontrado!')
    print(f'  Tipo: {type(prov).__name__}')
    print(f'  Atributos: {[a for a in dir(prov) if not a.startswith("_")]}')
    if hasattr(prov, 'page_no'): print(f'  page_no: {prov.page_no}')
    if hasattr(prov, 'bbox'): print(f'  bbox: {prov.bbox}')
    if hasattr(prov, 'charspan'): print(f'  charspan: {prov.charspan}')
else:
    print('table.prov NAO existe ou vazio')
    print(f'Atributos da tabela: {[a for a in dir(table) if not a.startswith("_")]}')

## 8. Extrair headers e rows (dados reais)

In [ ]:
if table is None:
    print('Sem tabela')
else:
    # Tentar export_to_dataframe (se existir)
    if hasattr(table, 'export_to_dataframe'):
        df = table.export_to_dataframe()
        print('export_to_dataframe() funcionou!')
        display(df.head(10))
    # Tentar table.data.table_cells
    elif hasattr(table, 'data') and hasattr(table.data, 'table_cells'):
        cells = table.data.table_cells
        print(f'table.data.table_cells: {len(cells)} celulas')
        max_row = max(c.start_row_offset_idx for c in cells)
        max_col = max(c.start_col_offset_idx for c in cells)
        print(f'Grid: {max_row+1} rows x {max_col+1} cols')
        print()
        # Montar grid
        for row_idx in range(min(max_row+1, 6)):  # Primeiras 5 linhas
            row_cells = sorted(
                [c for c in cells if c.start_row_offset_idx == row_idx],
                key=lambda c: c.start_col_offset_idx
            )
            texts = [c.text.strip() if c.text else '' for c in row_cells]
            prefix = 'HDR' if row_idx == 0 else f'R{row_idx}'
            print(f'{prefix}: {texts}')
    else:
        print('Nenhum metodo conhecido para extrair dados da tabela')
        print(f'Atributos: {[a for a in dir(table) if not a.startswith("_")]}')

## 9. Resumo das descobertas

Preencha apos rodar:

| Pergunta | Resposta |
|----------|----------|
| `doc.tables` existe? | |
| Tipo de tabela? | |
| `table.prov` existe? | |
| `table.data.table_cells` existe? | |
| `export_to_dataframe()` existe? | |
| Quantas tabelas no PDF AGU? | |
| Headers da 1a tabela? | |

Essas respostas corrigem `ptd_extracao.py`.